# Alec Bohm Career OPS Aging Projection

This notebook projects Alec Bohm's OPS at age 34 using MLB Stats API data. The baseline is Bohm's **career OPS**, not his current-season OPS, so the model is less reactive to a partial-season hot or cold stretch.

The model uses the requested factor order: age curve, comparable-player aging, plate discipline, batted-ball metrics, injury/durability, and pitcher/park context. The first four supplied weights already exceed 100%, so they are treated as priority scores and normalized after adding smaller injury and park/context scores.

To avoid hidden assumptions, the age curve is age-only. There is no manual position penalty. Bohm's position is fetched and displayed, but it is not used as an extra adjustment unless the comparable-player data naturally captures it.

In [ ]:
import datetime as dt
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

## MLB Stats API Helpers

In [ ]:
BASE_URL = "https://statsapi.mlb.com/api/v1"
PLAYER_NAME = "Alec Bohm"
TARGET_AGE = 34
CURRENT_YEAR = dt.date.today().year


def mlb_get(path, params=None):
    url = f"{BASE_URL}{path}"
    if params:
        url = f"{url}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=30) as response:
        return json.load(response)


def to_float(value):
    if value in (None, "", ".---", "-.--"):
        return np.nan
    return float(value)


def get_first_split(data):
    stats = data.get("stats", [])
    if not stats or not stats[0].get("splits"):
        return None
    return stats[0]["splits"][0]


def rate(numerator, denominator):
    return np.nan if denominator in (0, None) or pd.isna(denominator) else numerator / denominator


def flatten_hitting_split(split):
    stat = split.get("stat", {})
    player = split.get("player", {})
    position = split.get("position", {})
    team = split.get("team", {})
    pa = to_float(stat.get("plateAppearances"))
    ab = to_float(stat.get("atBats"))
    walks = to_float(stat.get("baseOnBalls"))
    strikeouts = to_float(stat.get("strikeOuts"))
    total_bases = to_float(stat.get("totalBases"))
    hits = to_float(stat.get("hits"))
    doubles = to_float(stat.get("doubles"))
    triples = to_float(stat.get("triples"))
    homers = to_float(stat.get("homeRuns"))
    singles = hits - doubles - triples - homers if pd.notna(hits) else np.nan

    return {
        "season": int(split.get("season", CURRENT_YEAR)),
        "player_id": player.get("id"),
        "name": player.get("fullName"),
        "team": team.get("name"),
        "position": position.get("abbreviation"),
        "age": to_float(stat.get("age")),
        "games": to_float(stat.get("gamesPlayed")),
        "pa": pa,
        "ab": ab,
        "avg": to_float(stat.get("avg")),
        "obp": to_float(stat.get("obp")),
        "slg": to_float(stat.get("slg")),
        "ops": to_float(stat.get("ops")),
        "bb_rate": rate(walks, pa),
        "k_rate": rate(strikeouts, pa),
        "iso": to_float(stat.get("slg")) - to_float(stat.get("avg")),
        "babip": to_float(stat.get("babip")),
        "go_ao": to_float(stat.get("groundOutsToAirouts")),
        "xbh_rate": rate(doubles + triples + homers, pa),
        "single_rate": rate(singles, pa),
        "total_bases": total_bases,
    }

## Fetch Bohm's Career Baseline

No OPS value is hardcoded. The player ID, age, position, team, and career hitting line are pulled from MLB Stats API.

In [ ]:
search = mlb_get("/people/search", {"names": PLAYER_NAME})
player = search["people"][0]
player_id = player["id"]

bio = mlb_get("/people", {"personIds": player_id, "hydrate": "currentTeam"})["people"][0]
career_split = get_first_split(mlb_get(f"/people/{player_id}/stats", {"stats": "career", "group": "hitting"}))
if career_split is None:
    raise ValueError(f"No career hitting line found for {PLAYER_NAME}")

career = flatten_hitting_split(career_split)
career.update(
    {
        "name": bio["fullName"],
        "age": bio["currentAge"],
        "team": bio.get("currentTeam", {}).get("name"),
        "position": bio.get("primaryPosition", {}).get("abbreviation"),
    }
)

career_ops = career["ops"]
current_age = int(career["age"])

pd.Series(career)[["name", "age", "team", "position", "games", "pa", "avg", "obp", "slg", "ops", "bb_rate", "k_rate", "iso", "babip", "go_ao"]]

## Weights

The weights are normalized from priority scores. This keeps the order exactly as requested while making the final model sum to 100%.

In [ ]:
weight_scores = {
    "age_curve": 30.0,
    "comparables": 27.5,
    "plate_discipline": 25.0,
    "batted_ball": 22.0,
    "injury_durability": 8.0,
    "pitcher_park_context": 4.0,
}

weights = pd.Series(weight_scores, dtype=float)
weights = weights / weights.sum()
weights.rename("normalized_weight").to_frame().assign(weight_pct=lambda x: x["normalized_weight"] * 100).round(3)

## Projection Components

The model uses formulas built from fetched MLB Stats API values. Comparable-player aging uses historical MLB season rows, pairing hitters at Bohm's current age with their age-34 seasons when both are available. No manual position penalty is applied.

In [ ]:
scenario_probs = pd.Series({"optimistic": 0.25, "median": 0.50, "pessimistic": 0.25})


def fetch_league_hitting(season, player_pool="all"):
    data = mlb_get(
        "/stats",
        {
            "stats": "season",
            "group": "hitting",
            "season": season,
            "playerPool": player_pool,
            "limit": 5000,
        },
    )
    splits = data.get("stats", [{}])[0].get("splits", [])
    rows = [flatten_hitting_split(split) for split in splits]
    return pd.DataFrame(rows)


def age_curve_component(start_age, target_age):
    years = np.arange(start_age + 1, target_age + 1)
    bat_speed_decline = np.where(years <= 31, 0.010, 0.026)
    median = np.prod(1 - bat_speed_decline)
    return pd.Series(
        {
            "optimistic": np.prod(1 - (bat_speed_decline * 0.70)),
            "median": median,
            "pessimistic": np.prod(1 - (bat_speed_decline * 1.45)),
        }
    ).clip(0.60, 1.02)


def comparable_component(start_age, target_age, career_line):
    seasons = range(max(2010, CURRENT_YEAR - 15), CURRENT_YEAR + 1)
    history = pd.concat([fetch_league_hitting(season) for season in seasons], ignore_index=True)
    history = history.dropna(subset=["player_id", "age", "ops", "pa", "bb_rate", "k_rate", "iso"])
    history = history[history["pa"] >= 150].copy()

    age_now = history[history["age"].round().astype(int).eq(start_age)].copy()
    age_target = history[history["age"].round().astype(int).eq(target_age)][["player_id", "ops"]].rename(columns={"ops": "target_ops"})
    paired = age_now.merge(age_target, on="player_id", how="inner")
    paired = paired[(paired["ops"] > 0) & (paired["target_ops"] > 0)].copy()

    if len(paired) < 20:
        return age_curve_component(start_age, target_age)

    features = ["ops", "bb_rate", "k_rate", "iso", "go_ao"]
    paired = paired.dropna(subset=features)
    target = pd.Series({feature: career_line[feature] for feature in features})
    scaled = (paired[features] - paired[features].mean()) / paired[features].std(ddof=0)
    target_scaled = (target - paired[features].mean()) / paired[features].std(ddof=0)
    paired["distance"] = np.sqrt(((scaled - target_scaled) ** 2).sum(axis=1))
    paired["retention"] = paired["target_ops"] / paired["ops"]

    comps = paired.nsmallest(min(40, len(paired)), "distance")
    return pd.Series(
        {
            "optimistic": comps["retention"].quantile(0.75),
            "median": comps["retention"].quantile(0.50),
            "pessimistic": comps["retention"].quantile(0.25),
        }
    ).clip(0.60, 1.05)


def plate_discipline_component(career_line, league):
    league_bb = league["bb_rate"].mean()
    league_k = league["k_rate"].mean()
    index = np.nanmean([career_line["bb_rate"] / league_bb, league_k / career_line["k_rate"]])
    median = np.clip(0.92 + 0.05 * (index - 1), 0.86, 0.99)
    return pd.Series({"optimistic": median + 0.035, "median": median, "pessimistic": median - 0.065}).clip(0.78, 1.03)


def batted_ball_component(career_line, league):
    iso_index = career_line["iso"] / league["iso"].mean()
    xbh_index = career_line["xbh_rate"] / league["xbh_rate"].mean()
    groundball_index = league["go_ao"].mean() / career_line["go_ao"]
    impact_index = np.nanmean([iso_index, xbh_index, groundball_index])
    median = np.clip(0.87 + 0.06 * (impact_index - 1), 0.78, 0.98)
    return pd.Series({"optimistic": median + 0.045, "median": median, "pessimistic": median - 0.085}).clip(0.70, 1.02)


def injury_durability_component(player_id):
    rows = []
    for season in range(max(2020, CURRENT_YEAR - 5), CURRENT_YEAR + 1):
        split = get_first_split(mlb_get(f"/people/{player_id}/stats", {"stats": "season", "group": "hitting", "season": season}))
        if split:
            rows.append(flatten_hitting_split(split))
    recent = pd.DataFrame(rows)
    if recent.empty:
        return pd.Series({"optimistic": 0.97, "median": 0.93, "pessimistic": 0.84})
    durability = recent["pa"].mean() / recent["pa"].max()
    median = np.clip(0.88 + 0.09 * durability, 0.88, 0.97)
    return pd.Series({"optimistic": median + 0.025, "median": median, "pessimistic": median - 0.095}).clip(0.76, 1.00)


def pitcher_park_component(player_id):
    splits = mlb_get(f"/people/{player_id}/stats", {"stats": "statSplits", "group": "hitting", "sitCodes": "h,r"})
    rows = [flatten_hitting_split(split) for split in splits.get("stats", [{}])[0].get("splits", [])]
    if len(rows) < 2:
        return pd.Series({"optimistic": 0.97, "median": 0.94, "pessimistic": 0.89})
    home_road = pd.DataFrame(rows)
    park_index = home_road["ops"].iloc[0] / home_road["ops"].iloc[1] if home_road["ops"].iloc[1] > 0 else 1.0
    median = np.clip(0.94 + 0.03 * (park_index - 1), 0.90, 0.98)
    return pd.Series({"optimistic": median + 0.020, "median": median, "pessimistic": median - 0.045}).clip(0.84, 1.01)

## Build Projection

In [ ]:
league_current = fetch_league_hitting(CURRENT_YEAR, player_pool="qualified")
league_current = league_current.dropna(subset=["ops", "bb_rate", "k_rate", "iso", "xbh_rate", "go_ao"])

components = {
    "age_curve": age_curve_component(current_age, TARGET_AGE),
    "comparables": comparable_component(current_age, TARGET_AGE, career),
    "plate_discipline": plate_discipline_component(career, league_current),
    "batted_ball": batted_ball_component(career, league_current),
    "injury_durability": injury_durability_component(player_id),
    "pitcher_park_context": pitcher_park_component(player_id),
}

component_modifiers = pd.DataFrame(components).round(3)
component_modifiers

In [ ]:
weighted_multiplier = pd.DataFrame(components).mul(weights, axis=1).sum(axis=1)
projection = pd.DataFrame(
    {
        "scenario": weighted_multiplier.index,
        "probability": scenario_probs.loc[weighted_multiplier.index],
        "weighted_multiplier": weighted_multiplier,
    }
).reset_index(drop=True)

projection["projected_ops_age_34"] = career_ops * projection["weighted_multiplier"]
projection["probability_weighted_ops"] = projection["probability"] * projection["projected_ops_age_34"]
expected_ops = projection["probability_weighted_ops"].sum()

projection.round(3)

In [ ]:
print(f"Player: {career['name']} ({career['position']})")
print(f"Career OPS from MLB Stats API: {career_ops:.3f}")
print(f"Projected weighted expected OPS at age {TARGET_AGE}: {expected_ops:.3f}")

## Scenario Distribution

In [ ]:
labels = {"optimistic": "Optimistic", "median": "Median", "pessimistic": "Pessimistic"}
colors = {"optimistic": "#2E7D32", "median": "#1565C0", "pessimistic": "#C62828"}

fig, ax = plt.subplots(figsize=(10, 6))
x_labels = [labels[s] for s in projection["scenario"]]
bars = ax.bar(
    x_labels,
    projection["projected_ops_age_34"],
    color=[colors[s] for s in projection["scenario"]],
    alpha=0.88,
    width=0.62,
)

ax.axhline(expected_ops, color="#111111", linestyle="--", linewidth=2, label=f"Expected OPS: {expected_ops:.3f}")

for bar, row in zip(bars, projection.itertuples(index=False)):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        row.projected_ops_age_34 + 0.010,
        f"{row.projected_ops_age_34:.3f}\n{row.probability:.0%}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.text(len(projection) - 0.10, expected_ops + 0.006, f"Expected: {expected_ops:.3f}", ha="right", va="bottom", fontsize=11)
ax.set_title(f"{career['name']} Age-{TARGET_AGE} OPS Projection", fontsize=16, fontweight="bold", pad=16)
ax.set_ylabel("Projected OPS")
ax.set_xlabel("Scenario")
ax.set_ylim(max(0.45, projection["projected_ops_age_34"].min() - 0.06), min(0.95, projection["projected_ops_age_34"].max() + 0.08))
ax.legend(loc="lower left", frameon=True)
plt.tight_layout()
plt.show()

## Interpretation

This projection uses Bohm's career OPS as the starting point, which makes the baseline more stable than a single partial-season OPS. The model then applies aging pressure from age 29 to 34, with attention to the post-31 bat-speed decline window.

The key risk is batted-ball erosion: if Bohm loses impact without gaining walk-rate cushion, his OPS can settle closer to the pessimistic range. The key upside driver is contact and plate-discipline stability. If those skills hold and the comparable-player path is friendly, he can remain a playable regular-level bat at 34 rather than sliding into a bench-only offensive profile.